# BanglaBox voice clone test (mimnets-bangla-voice, Phase 1)

Clones **your** voice from a 10 to 15 s reference clip with **BanglaBox** (a Bangla fine-tune of Chatterbox TTS) and reads the project's three test scripts.

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.
Then run the cells **one by one from top to bottom** (click ▶ on each, wait for it to finish).

You'll need: `reference.wav` from `voice-clone-lab/reference/` on your PC (made by `record_reference.py`).

Credits: model by the BanglaBox authors (https://huggingface.co/Banglabox/chatterbox-bangla-tts, arXiv 2610.09211), built on Chatterbox by Resemble AI (MIT). Only clone voices you have permission to use.

## 1. Check the GPU
You should see `Tesla T4`. If you see `NO GPU`, change the runtime type (see above) and run this cell again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: Runtime > Change runtime type > T4 GPU"
import sys; print("Python", sys.version.split()[0])

## 2. Install the libraries (about 3 to 5 minutes)
These are the exact versions BanglaBox lists in its `inference/requirements.txt`, plus an older `huggingface_hub` that `diffusers 0.29` needs.

- Red lines saying *"pip's dependency resolver does not currently take into account..."* (about gradio, torchvision) are normal on Colab. Ignore them.
- If Colab pops up **"Restart session"**, click it, then continue with step 3 (don't re-run step 2).

In [ ]:
!pip install -q torch==2.6.0 torchaudio==2.6.0 accelerate==1.14.0 conformer==0.3.2 diffusers==0.29.0 \
    einops==0.8.2 librosa==0.11.0 numpy==2.1.2 omegaconf==2.3.1 peft==0.17.1 s3tokenizer==0.3.0 \
    safetensors==0.5.3 soundfile==0.13.1 tokenizers==0.20.3 transformers==4.46.3 \
    huggingface_hub==0.25.2 bnnumerizer bnunicodenormalizer
# Colab's torchvision is built for a newer torch and can crash transformers; we don't need it.
!pip uninstall -y -q torchvision
print("Install finished.")

## 3. Download BanglaBox (about 3.5 GB, 2 to 4 minutes)
The model repo keeps the base model in `pretrained_models/` and the Bangla adapter in `checkpoint/adapter/`, but its `infer.py` looks for both next to itself. This cell adds links so it finds them.

In [ ]:
import os
from huggingface_hub import snapshot_download

ROOT = snapshot_download("Banglabox/chatterbox-bangla-tts", local_dir="/content/banglabox",
                         allow_patterns=["pretrained_models/*", "checkpoint/*", "inference/*", "voices/*"])
KIT = os.path.join(ROOT, "inference")
for name, target in [("pretrained_models", "../pretrained_models"), ("adapter", "../checkpoint/adapter")]:
    link = os.path.join(KIT, name)
    if not os.path.exists(link):
        os.symlink(target, link)
for f in ["infer.py", "NEW_VOCAB_SIZE.txt", "pretrained_models/t3_cfg.safetensors", "adapter/adapter_config.json"]:
    print(("OK      " if os.path.exists(os.path.join(KIT, f)) else "MISSING ") + f)

## 4. Get our project code (text normaliser + test scripts)
Clones https://github.com/mimnets/mimnets-bangla-voice. If the repo is **private** the clone fails; then run the next cell instead and upload `bangla_text.py` and the three files from `voice-clone-lab/test_scripts/`.

In [ ]:
!rm -rf /content/project && git clone -q --depth 1 https://github.com/mimnets/mimnets-bangla-voice.git /content/project \
  && echo "Cloned OK" || echo "Clone failed: run the upload cell below"

In [ ]:
# ONLY if the clone above failed: upload bangla_text.py and the test_scripts/*.txt files.
import os, shutil
from google.colab import files
LAB = "/content/project/voice-clone-lab"
os.makedirs(f"{LAB}/test_scripts", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"{LAB}/{name}" if name.endswith(".py") else f"{LAB}/test_scripts/{name}")
print(os.listdir(LAB), os.listdir(f"{LAB}/test_scripts"))

## 5. Upload your reference clip
Click **Choose files** and pick `reference.wav` from `voice-clone-lab\reference\` on your PC. Listen to the player to check it's the right clip.

In [ ]:
from google.colab import files
from IPython.display import Audio, display
uploaded = files.upload()
REF = "/content/" + next(n for n in uploaded if n.lower().endswith(".wav"))
print("Reference:", REF)
display(Audio(REF))

## 6. Load the model (1 to 2 minutes)

In [ ]:
import sys, time, torch
sys.path.insert(0, KIT)
sys.path.insert(0, "/content/project/voice-clone-lab")
os.chdir(KIT)
from infer import BanglaTTS
from bangla_text import prepare

t = time.time()
tts = BanglaTTS(KIT)
print(f"Loaded in {time.time() - t:.0f} s on", "GPU" if torch.cuda.is_available() else "CPU (slow!)", "| sample rate:", tts.sr)

## 7. Quick test: one sentence

In [ ]:
import numpy as np

def to_np(wav):
    if hasattr(wav, "detach"):
        wav = wav.detach().cpu().numpy()
    return np.asarray(wav, dtype=np.float32).squeeze()

torch.manual_seed(0)
t = time.time()
wav, sr = tts.tts("আসসালামু আলাইকুম, আমি এখন আমার নিজের কণ্ঠে বাংলা বলছি।", REF)
wav = to_np(wav)
print(f"{len(wav) / sr:.1f} s of audio in {time.time() - t:.1f} s")
display(Audio(wav, rate=sr))

## 8. Read the three test scripts
Each script goes through our `bangla_text.prepare()` (numbers → Bangladeshi words, split into sentences), then each sentence is synthesised and joined with a short pause. BanglaBox also runs its own normaliser inside `tts()`.

Speed is printed as "x real time": above 1.0 means faster than the audio's length.

In [ ]:
import glob, datetime
import soundfile as sf

PAUSE_S = 0.25     # silence between sentences
SEED = 0           # change to get a different "take"
OUT = "/content/out"
os.makedirs(OUT, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

def speak(text, label):
    torch.manual_seed(SEED)
    sentences = prepare(text, max_chars=200)
    pieces, t0 = [], time.time()
    for i, s in enumerate(sentences, 1):
        print(f"  {i}/{len(sentences)}  {s}")
        wav, sr = tts.tts(s, REF)
        pieces += [to_np(wav), np.zeros(int(sr * PAUSE_S), np.float32)]
    audio = np.concatenate(pieces)
    took = time.time() - t0
    path = f"{OUT}/banglabox_{label}_{stamp}.wav"
    sf.write(path, audio, sr)
    print(f"Saved {path}: {len(audio) / sr:.1f} s audio in {took:.0f} s ({len(audio) / sr / took:.2f}x real time)\n")
    display(Audio(audio, rate=sr))

for script in sorted(glob.glob("/content/project/voice-clone-lab/test_scripts/*.txt")):
    label = os.path.basename(script)[:-4]
    print("==", label)
    speak(open(script, encoding="utf-8").read(), label)

## 9. Try your own text (optional)
Edit `TEXT` and run the cell. Change `SEED` above to get a different take of the same text.

In [ ]:
TEXT = "বন্ধুরা, আজকের ভিডিওটা শেষ পর্যন্ত দেখো। দাম মাত্র ৳৯৯৯!"
speak(TEXT, "custom")

## 10. Download all results
Downloads `banglabox_out.zip`. Put the WAVs in `voice-clone-lab\out\` on your PC (that folder is git-ignored) and fill in `RESULTS.md`.

In [ ]:
!cd /content && zip -qr banglabox_out.zip out && echo "Zipped"
from google.colab import files
files.download("/content/banglabox_out.zip")